In [1]:
import torch
import cv2
import numpy as np
import pandas as pd
import os
import gc
import sys
from tqdm import tqdm
from segment_anything import sam_model_registry, SamPredictor
import requests

# --- LOCAL MODULES ---
sys.path.append('..')
from src.utils import get_device
from src.features import get_house_mask, get_vegetation_layers, compute_metrics, create_stack

# --- 1. HARDWARE SETUP ---
DEVICE = get_device()

# --- 2. MODEL PATH ---
CHECKPOINT_PATH = "../models/sam_vit_b_01ec64.pth"
MODEL_URL = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"

if not os.path.exists(CHECKPOINT_PATH):
    print("Downloading SAM weights...")
    os.makedirs("../models", exist_ok=True)
    response = requests.get(MODEL_URL)
    with open(CHECKPOINT_PATH, "wb") as f:
        f.write(response.content)
    print("Download complete.")

# --- 3. LOAD MODEL ---
print("Loading SAM model to GPU...")
sam = sam_model_registry["vit_b"](checkpoint=CHECKPOINT_PATH)
sam.to(device=DEVICE)
predictor = SamPredictor(sam)
print("Model loaded and ready.")

/Users/tejusagrawal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✅ Using Apple MPS (Neural Engine)
   PyTorch Version: 2.8.0
Loading SAM model to GPU...
Model loaded and ready.


In [2]:
# --- CONFIGURATION ---
IMAGE_DIR = "../data/images"
INPUT_CSV = "../data/processed/clean_homes.csv"
OUTPUT_CSV = "../data/processed/final_dataset.csv"
TENSOR_DIR = "../data/mask_tensors"

os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
os.makedirs(TENSOR_DIR, exist_ok=True)

BATCH_SIZE = 500

# 1. Map Existing Progress
# A. Check CSV
csv_ids = set()
if os.path.exists(OUTPUT_CSV):
    try:
        existing_df = pd.read_csv(OUTPUT_CSV)
        csv_ids = set(existing_df['id'].astype(str))
        print(f"📋 Found {len(csv_ids)} entries in CSV.")
    except:
        print("⚠️ CSV corrupt. Starting fresh.")
else:
    # Init Headers
    cols = ['id', 'target', 'lat', 'lon', 
            'structure_area_m2', 'tree_area_m2', 'grass_area_m2', 
            'tree_count', 'defensible_space_m', 'compactness']
    pd.DataFrame(columns=cols).to_csv(OUTPUT_CSV, index=False)
    print("🚀 Created new CSV.")

# B. Check Tensors
existing_tensors = set()
if os.path.exists(TENSOR_DIR):
    files = os.listdir(TENSOR_DIR)
    for f in files:
        if f.endswith('.npy'):
            existing_tensors.add(f.split('_')[0])
    print(f"📦 Found {len(existing_tensors)} tensors.")

# 2. Main Loop
if not os.path.exists(INPUT_CSV):
    print(f"⚠️ Input CSV not found at {INPUT_CSV}. Please run data_cleaning.ipynb first.")
else:
    df_meta = pd.read_csv(INPUT_CSV)
    df_meta['id_str'] = df_meta['id'].astype(str)
    results_buffer = []

    print(f"🔄 Scanning {len(df_meta)} homes...")

    for i, row in tqdm(df_meta.iterrows(), total=len(df_meta), unit="img"):
        uid = row['id_str']
        
        # Smart Checks
        needs_csv = uid not in csv_ids
        needs_tensor = uid not in existing_tensors
        
        # Optimization: Skip if both exist
        if not needs_csv and not needs_tensor:
            continue

        # Path check
        fname = f"{row['filename']}.jpg" if 'filename' in row and pd.notna(row['filename']) else f"{row['id']}.jpg"
        img_path = os.path.join(IMAGE_DIR, fname)
        
        if not os.path.exists(img_path):
            continue

        try:
            # A. Run Pipeline
            image = cv2.imread(img_path)
            if image is None: continue
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Run Encoder (GPU Heavy Lift)
            predictor.set_image(image_rgb)
            
            # Get Masks
            house_mask = get_house_mask(predictor, image_rgb)
            tree_mask, grass_mask = get_vegetation_layers(image_rgb, house_mask)

            # B. Task 1: Generate CSV Data
            if needs_csv:
                feats = compute_metrics(house_mask, tree_mask, grass_mask)
                if feats:
                    record = {
                        'id': row['id'], 'target': row['target'],
                        'lat': row['lat'], 'lon': row['lon'],
                        **feats
                    }
                    results_buffer.append(record)
                    csv_ids.add(uid)

            # C. Task 2: Generate Tensor
            if needs_tensor:
                stack = create_stack(house_mask, tree_mask)
                save_name = f"{row['id']}_{int(row['target'])}.npy"
                np.save(os.path.join(TENSOR_DIR, save_name), stack)

        except Exception as e:
            print(f"⚠️ Error on {fname}: {e}")
            continue

        # D. Batch Save (CSV)
        if len(results_buffer) >= BATCH_SIZE:
            pd.DataFrame(results_buffer).to_csv(OUTPUT_CSV, mode='a', header=False, index=False)
            results_buffer = []
            gc.collect()

    # Final Flush
    if results_buffer:
        pd.DataFrame(results_buffer).to_csv(OUTPUT_CSV, mode='a', header=False, index=False)

    print("\n✅ SYNC COMPLETE.")

📋 Found 21361 entries in CSV.
📦 Found 20940 tensors.
🔄 Scanning 21361 homes...


100%|██████████| 21361/21361 [00:00<00:00, 79659.48img/s]


✅ SYNC COMPLETE.
